<div align="center">
<h1>🔐 KRİPTOQRAFİYA KURSU</h1>
<h2>Məşğələ 17</h2>
<h2>HMAC və Poly1305</h2>
<h3 style="color: #8B4513;">Heş əsaslı və universal MAK-lar</h3>
<br>
<h3>Məşğələ vaxtı: 2 saat</h3>
<br>
<p><em>Hazırlanma tarixi: 2024</em></p>
</div>

## 📑 Mündəricat

1. [Məşğələnin məqsədləri](#məşğələnin-məqsədləri)
2. [Lazım olan kitabxanalar](#lazım-olan-kitabxanalar)
3. [Hazırlıq (15 dəq)](#hazırlıq-15-dəq)
4. [Xatırlatma: MAK nədir? (5 dəq)](#xatırlatma-mak-nədir-5-dəq)
5. [HMAC (Hash-based Message Authentication Code) (25 dəq)](#hmac-hash-based-message-authentication-code-25-dəq)
6. [Poly1305 (20 dəq)](#poly1305-20-dəq)
7. [ChaCha20-Poly1305 AEAD rejimi (20 dəq)](#chacha20-poly1305-aead-rejimi-20-dəq)
8. [HMAC və Poly1305 müqayisəsi (10 dəq)](#hmac-və-poly1305-müqayisəsi-10-dəq)
9. [İnteqrasiya edilmiş tətbiq (15 dəq)](#inteqrasiya-edilmiş-tətbiq-15-dəq)
10. [Ev tapşırığı](#ev-tapşırığı)
11. [Yekun və müzakirə sualları](#yekun-və-müzakirə-sualları)
12. [Əlavə resurslar](#əlavə-resurslar)

## 🎯 Məşğələnin məqsədləri

Bu məşğələni bitirdikdən sonra siz:

✅ HMAC-ın quruluşunu, iş prinsipini və təhlükəsizlik xüsusiyyətlərini izah edə biləcəksiniz  
✅ HMAC-ın uzunluq genişləndirmə hücumuna qarşı davamlılığını başa düşəcəksiniz  
✅ Universal heşləmə anlayışını və Poly1305-in riyazi əsaslarını izah edə biləcəksiniz  
✅ ChaCha20-Poly1305 AEAD rejiminin iş prinsipini və üstünlüklərini başa düşəcəksiniz  
✅ HMAC və Poly1305-in Python-da implementasiyasını yaza biləcəksiniz  
✅ Müxtəlif MAK alqoritmlərini müqayisə edə və tətbiq üçün uyğun olanı seçə biləcəksiniz

## 📚 Lazım olan kitabxanalar

| Kitabxana | Quraşdırma | İstifadə sahəsi |
|-----------|------------|-----------------|
| hmac | Python standart kitabxanası | HMAC hesablamaları |
| hashlib | Python standart kitabxanası | SHA-2, SHA-3 heş funksiyaları |
| cryptography | `!pip install cryptography` | ChaCha20-Poly1305, AEAD |
| PyCryptodome | `!pip install pycryptodome` | Poly1305, ChaCha20 |

> 💡 **Qeyd:** HMAC və hashlib Python-un standart kitabxanasına daxildir. Cryptography və PyCryptodome isə əlavə quraşdırma tələb edir.

In [ ]:
# Lazımi kitabxanaların quraşdırılması

!pip install cryptography pycryptodome --quiet

print("✅ Kitabxanalar quraşdırıldı")

## 🔧 Hazırlıq (15 dəq)

### 3.1 Python mühitinin yoxlanılması

Aşağıdakı kodu işə salaraq lazımi modulların yükləndiyini yoxlayın:

In [ ]:
import sys
import hmac
import hashlib
import os
import time
import math
import random
from collections import Counter

# cryptography kitabxanası
try:
    from cryptography.hazmat.primitives.ciphers.aead import ChaCha20Poly1305
    from cryptography.hazmat.primitives import hashes
    from cryptography.hazmat.primitives.kdf.hkdf import HKDF
    print("✅ cryptography yüklü - ChaCha20-Poly1305 istifadə edilə bilər")
    CRYPTOGRAPHY_AVAILABLE = True
except ImportError:
    print("⚠️ cryptography yoxdur - pip install cryptography edin")
    CRYPTOGRAPHY_AVAILABLE = False

# PyCryptodome kitabxanası
try:
    from Crypto.Cipher import ChaCha20
    from Crypto.Random import get_random_bytes
    print("✅ PyCryptodome yüklü - ChaCha20 istifadə edilə bilər")
    CRYPTO_AVAILABLE = True
except ImportError:
    print("⚠️ PyCryptodome yoxdur - pip install pycryptodome edin")
    CRYPTO_AVAILABLE = False

print(f"\n🐍 Python versiyası: {sys.version}")

### 3.2 İşçi qovluğun yaradılması

In [ ]:
import os
from pathlib import Path

# Notebook-un kök qovluğunu yalnız ilk icrada yadda saxla
if 'NOTEBOOK_ROOT' not in globals():
    NOTEBOOK_ROOT = Path.cwd().resolve()

# İşçi qovluğu yaradaq (təkrar icrada iç-içə qovluq yaranmasın)
workspace = NOTEBOOK_ROOT / "crypto_workshop" / "lecture17"
workspace.mkdir(parents=True, exist_ok=True)
os.chdir(workspace)

print(f"📁 İşçi qovluq: {os.getcwd()}")
print(f"📂 Qovluğun məzmunu: {os.listdir('.')}")

## 📖 Xatırlatma: MAK nədir? (5 dəq)

<div style="background-color: #f0f8ff; padding: 15px; border-radius: 10px; border-left: 5px solid #4682b4;">
<h4>🔑 MAK (Message Authentication Code)</h4>
<p>MAK — mesajın tamlığını və mənbə autentifikasiyasını təmin edən kriptoqrafik mexanizmdir.</p>

<p style="text-align: center; font-size: 1.2em;">$$\text{MAC}_K(M) = T$$</p>

<p>burada $K$ gizli açar, $M$ mesaj, $T$ isə autentifikasiya teqidir.</p>
</div>

<div style="background-color: #e8f5e9; padding: 15px; border-radius: 10px; border-left: 5px solid #4caf50; margin-top: 10px;">
<h4>📌 Bu məşğələdə iki mühüm MAK alqoritmi</h4>
<ul>
    <li><b>HMAC:</b> Heş funksiyaları əsasında qurulan MAK</li>
    <li><b>Poly1305:</b> Universal heşləmə əsaslı yüksək sürətli MAK</li>
</ul>
</div>

## 🔏 HMAC (Hash-based Message Authentication Code) (25 dəq)

<div style="background-color: #f0f8ff; padding: 15px; border-radius: 10px; border-left: 5px solid #4682b4;">
<h4>🔏 HMAC</h4>
<p>HMAC aşağıdakı kimi təyin olunur:</p>

<p style="text-align: center; font-size: 1.2em;">$$\text{HMAC}_K(M) = H\big( (K \oplus \text{opad}) \parallel H( (K \oplus \text{ipad}) \parallel M ) \big)$$</p>

<p>burada:</p>
<ul>
    <li>$H$ — əsas heş funksiyası (məsələn, SHA-256)</li>
    <li>$K$ — gizli açar</li>
    <li>$\text{ipad} = 0x36$ (təkrarlanan daxili doldurma)</li>
    <li>$\text{opad} = 0x5C$ (təkrarlanan xarici doldurma)</li>
    <li>$\parallel$ — birləşdirmə</li>
    <li>$\oplus$ — XOR</li>
</ul>
</div>

In [ ]:
def hmac_demo():
    """
    Python hmac modulu ilə HMAC nümayişi
    """
    import secrets

    print("=" * 70)
    print("🔏 HMAC (HASH-BASED MAC) NÜMAYİŞİ")
    print("=" * 70)

    # Gizli açar: notebookda sərt kodlaşdırmaq əvəzinə hər işə salmada yaradılır.
    # Real sistemlərdə açarı çap etməyin və təhlükəsiz açar idarəetməsindən istifadə edin.
    key = secrets.token_bytes(32)
    message = b'Bu vacib mesajdir'

    print(f"🔑 Açar: {key}")
    print(f"📨 Mesaj: {message}")

    # HMAC-SHA256 hesabla
    h = hmac.new(key, message, hashlib.sha256)
    mac = h.hexdigest()

    print(f"\n🔏 HMAC-SHA256: {mac}")
    print(f"   Uzunluq: {len(mac)} hex simvol = {len(mac)*4} bit")

    # Fərqli heş funksiyaları ilə HMAC
    print("\n📊 Müxtəlif heş funksiyaları:")
    for algo in ['sha1', 'sha256', 'sha384', 'sha512', 'sha3_256']:
        if hasattr(hashlib, algo):
            h = hmac.new(key, message, getattr(hashlib, algo))
            print(f"   HMAC-{algo.upper()}: {h.hexdigest()[:16]}...")

    return key, message, mac

key, msg, mac = hmac_demo()


### 5.1 HMAC vs $H(K || M)$ müqayisəsi

<div style="background-color: #ffebee; padding: 15px; border-radius: 10px; border-left: 5px solid #f44336;">
<h4>⚠️ Uzunluq genişləndirmə hücumu</h4>
<p>$H(K || M)$ konstruksiyası uzunluq genişləndirmə hücumuna qarşı həssasdır. HMAC bu hücuma qarşı davamlıdır.</p>
</div>

In [ ]:
def length_extension_warning_demo():
    """
    Uzunluq genişləndirmə hücumu haqqında xəbərdarlıq.

    Qeyd: Bu nümunə real saxtalaşdırılmış mesaj/tag qurmur; yalnız
    H(K || M) konstruksiyasının HMAC ilə müqayisədə niyə riskli olduğunu göstərir.
    """
    print("\n" + "-" * 70)
    print("⚠️ UZUNLUQ GENİŞLƏNDİRMƏ XƏBƏRDARLIĞI")
    print("-" * 70)

    key = b'gizli'
    message = b'Mesaj'

    # Sadə H(K || M) konstruksiyası
    simple_mac = hashlib.sha256(key + message).hexdigest()

    # HMAC
    h = hmac.new(key, message, hashlib.sha256)
    hmac_mac = h.hexdigest()

    print(f"🔑 Açar: {key}")
    print(f"📨 Mesaj: {message}")
    print(f"\n📌 Sadə H(K||M): {simple_mac}")
    print(f"🔏 HMAC: {hmac_mac}")

    print("\n🚨 Sadə H(K||M) uzunluq genişləndirmə hücumuna HƏSSASDIR!")
    print("✅ HMAC isə bu hücuma qarşı DAVAMLIDIR.")
    print("ℹ️ Real hücum nümayişi üçün SHA-256 daxili vəziyyətini və təxmin edilən açar uzunluğunu qurmaq lazımdır.")

length_extension_warning_demo()


### 5.2 Öz HMAC implementasiyamız

In [ ]:
def hmac_implementation(key, message, hash_func=hashlib.sha256, block_size=None):
    """
    HMAC-ın öz implementasiyası (təhsil məqsədli)
    """
    key = bytes(key)
    message = bytes(message)

    # HMAC blok ölçüsü seçilən hash funksiyasından asılıdır
    if block_size is None:
        block_size = hash_func().block_size

    # Açarı blok ölçüsünə uyğunlaşdır
    if len(key) > block_size:
        key = hash_func(key).digest()
    if len(key) < block_size:
        key = key + b'\x00' * (block_size - len(key))

    # ipad və opad
    ipad = bytes([0x36] * block_size)
    opad = bytes([0x5C] * block_size)

    # Daxili və xarici XOR
    key_ipad = bytes(a ^ b for a, b in zip(key, ipad))
    key_opad = bytes(a ^ b for a, b in zip(key, opad))

    # HMAC = H((K ⊕ opad) || H((K ⊕ ipad) || M))
    inner_hash = hash_func(key_ipad + message).digest()
    return hash_func(key_opad + inner_hash).digest()

# Test
print("\n" + "-" * 70)
print("🔧 ÖZ HMAC IMPLEMENTASİYAMIZ")
print("-" * 70)

key = b'gizli'
message = b'Test mesaji'

our_hmac = hmac_implementation(key, message).hex()
py_hmac = hmac.new(key, message, hashlib.sha256).hexdigest()

print(f"🔑 Açar: {key}")
print(f"📨 Mesaj: {message}")
print(f"\n🛠️ Öz HMAC:    {our_hmac}")
print(f"🐍 Python HMAC: {py_hmac}")
print(f"✅ Bərabərdir? {our_hmac == py_hmac}")

print("\n🔍 Əlavə uyğunluq testləri:")
validation_key = b'K' * 200
validation_message = b'M' * 50
for algo in [hashlib.sha1, hashlib.sha256, hashlib.sha512, hashlib.sha3_256]:
    ours = hmac_implementation(validation_key, validation_message, algo).hex()
    ref = hmac.new(validation_key, validation_message, algo).hexdigest()
    print(f"   {algo().name:<8} -> {ours == ref}")

### 5.3 HKDF (HMAC-based Key Derivation Function)

In [ ]:
def hkdf_extract(salt, ikm, hash_func=hashlib.sha256):
    """
    HKDF-Extract mərhələsi
    """
    return hmac.new(salt, ikm, hash_func).digest()


def hkdf_expand(prk, info=b'', length=32, hash_func=hashlib.sha256):
    """
    HKDF-Expand mərhələsi
    """
    hash_len = hash_func().digest_size
    n = math.ceil(length / hash_len)

    if n > 255:
        raise ValueError("❌ HKDF üçün maksimum uzunluq 255 * HashLen olmalıdır")

    okm = b''
    t = b''

    for i in range(1, n + 1):
        t = hmac.new(prk, t + info + bytes([i]), hash_func).digest()
        okm += t

    return okm[:length]


def hkdf_demo():
    """
    HKDF (HMAC-based Key Derivation Function) nümayişi
    """
    print("\n" + "-" * 70)
    print("🔑 HKDF (HMAC-BASED KEY DERIVATION FUNCTION)")
    print("-" * 70)

    # İlkin açar materialı
    ikm = b'gizli parol'
    salt = os.urandom(16)
    info = b'context info'
    length = 32

    print(f"📦 İlkin açar materialı (IKM): {ikm}")
    print(f"🧂 Salt: {salt.hex()}")
    print(f"ℹ️ Info: {info}")
    print(f"🎯 Hədəf uzunluq: {length} bayt")

    # HKDF-nin iki mərhələsi
    prk = hkdf_extract(salt, ikm, hashlib.sha256)
    okm = hkdf_expand(prk, info, length=length, hash_func=hashlib.sha256)

    print(f"\n🔑 PRK (pseudorandom key): {prk.hex()}")
    print(f"🔐 OKM (output keying material): {okm.hex()}")
    print(f"📏 OKM uzunluğu: {len(okm)} bayt")

    if CRYPTOGRAPHY_AVAILABLE:
        from cryptography.hazmat.primitives.kdf.hkdf import HKDF
        from cryptography.hazmat.primitives import hashes

        hkdf = HKDF(
            algorithm=hashes.SHA256(),
            length=length,
            salt=salt,
            info=info,
        )
        key = hkdf.derive(ikm)
        print(f"\n📚 Cryptography HKDF: {key.hex()}")
        print(f"✅ Eynidir? {key == okm}")

hkdf_demo()

### ✍️ Çalışma 1: HMAC (1.5 bal)

Aşağıdakı tapşırıqları yerinə yetirin:

1. **Şəxsi HMAC:** HMAC-SHA256 istifadə edərək öz adınız üçün MAK yaradın.

2. **Nəzəri sual:** $H(K || M)$ konstruksiyasının niyə təhlükəsiz olmadığını izah edin.

3. **Implementasiya müqayisəsi:** Öz HMAC implementasiyanızı yazın və onu Python hmac modulu ilə müqayisə edin.

4. **HKDF:** HKDF istifadə edərək verilmiş giriş açar materialından 32 baytlıq açar törədin. Əgər məqsəd həqiqətən təsadüfi açar yaratmaqdırsa, `os.urandom(32)` və ya `secrets.token_bytes(32)` istifadə edin; parol əsaslı giriş üçün isə əvvəlcə Argon2, scrypt, PBKDF2 və ya bcrypt kimi parol KDF-i istifadə olunmalıdır.

In [ ]:
# Çalışma 1 - Cavablar

print("📝 ÇALIŞMA 1 CAVABLARI")
print("=" * 80)

# 1. Şəxsi HMAC
print("\n1. ŞƏXSİ HMAC:")
my_name = "Səməd Vəkilov"  # Öz adınızla əvəz edin
key = b'gizli_ortaq_acar'
h = hmac.new(key, my_name.encode(), hashlib.sha256)
print(f"   Ad: {my_name}")
print(f"   HMAC-SHA256: {h.hexdigest()}")

# 2. Nəzəri sual
print("\n2. NƏZƏRİ SUAL:")
print("""
   H(K || M) konstruksiyası təhlükəsiz deyil, çünki:
   • Merkl-Damqord quruluşlu heş funksiyalarında uzunluq genişləndirmə hücumuna həssasdır
   • H(K || M) və len(M) məlumdursa, H(K || M || pad || X) hesablamaq olar
   • Bu, hücumçuya mesajı dəyişdirməyə imkan verir
   • HMAC ipad/opad istifadə edərək bu problemi həll edir
""")

# 3. Implementasiya müqayisəsi (artıq yuxarıda edildi)
print("\n3. IMPLEMENTASİYA MÜQAYİSƏSİ:")
print("   Yuxarıdakı nümayişdə öz implementasiyamız Python modulu ilə uyğun gəldi.")

# 4. HKDF ilə açar yaratma
print("\n4. HKDF AÇAR YARATMA:")
ikm = "gizli parol materialı".encode("utf-8")
salt = os.urandom(16)
info = b'my app context'
length = 32

prk = hkdf_extract(salt, ikm, hashlib.sha256)
okm = hkdf_expand(prk, info, length=length, hash_func=hashlib.sha256)

print(f"   Yaranan {length} baytlıq açar: {okm.hex()}")

## 🔢 Poly1305 (20 dəq)

<div style="background-color: #f0f8ff; padding: 15px; border-radius: 10px; border-left: 5px solid #4682b4;">
<h4>🎲 Universal heşləmə anlayışı</h4>
<p>$H = \{h: D \rightarrow R\}$ heş funksiyaları ailəsi <b>universaldır</b>, əgər ixtiyari iki fərqli $x, y \in D$ girişi üçün, təsadüfi seçilmiş $h \in H$ funksiyasının $h(x) = h(y)$ olma ehtimalı $\varepsilon \leq 1/|R|$ -dən böyük deyilsə.</p>
</div>

<div style="background-color: #f0f8ff; padding: 15px; border-radius: 10px; border-left: 5px solid #4682b4; margin-top: 10px;">
<h4>🔢 Poly1305-in riyazi əsasları</h4>
<p>Poly1305 $\mathbb{F}_{2^{130}-5}$ sadə meydanında işləyir:</p>
<p style="text-align: center; font-size: 1.2em;">$$p = 2^{130} - 5$$</p>
</div>

In [ ]:
class Poly1305:
    """
    Poly1305 universal MAK implementasiyası
    Bernstein-in spesifikasiyasına uyğun təhsil məqsədli versiya
    """

    def __init__(self, key):
        """
        key: 32 bayt (256 bit) - r (16 bayt) + s (16 bayt)
        """
        if len(key) != 32:
            raise ValueError("❌ Açar 32 bayt olmalıdır")

        # r açarını ayır və məhdudlaşdır
        self.r = int.from_bytes(key[:16], 'little')
        self._clamp_r()

        # s açarını ayır
        self.s = int.from_bytes(key[16:], 'little')

        self.p = (1 << 130) - 5

    def _clamp_r(self):
        """
        r açarını məhdudlaşdır - təhlükəsizlik üçün
        r = r & 0x0ffffffc0ffffffc0ffffffc0fffffff
        """
        mask = 0x0ffffffc0ffffffc0ffffffc0fffffff
        self.r &= mask

    def _block_to_int(self, block):
        """
        Hər bloka RFC-yə uyğun olaraq 0x01 əlavə edilir
        """
        return int.from_bytes(block + b'\x01', 'little')

    def hash(self, data):
        """
        Poly1305 hesabla
        """
        data = bytes(data)
        acc = 0

        for i in range(0, len(data), 16):
            block = data[i:i+16]
            n = self._block_to_int(block)
            acc = (acc + n) * self.r % self.p

        # s əlavə et və mod 2^128
        acc = (acc + self.s) % (1 << 128)
        return acc.to_bytes(16, 'little')

    def hexdigest(self, data):
        """
        Poly1305 teqini hex formatda qaytar
        """
        return self.hash(data).hex()

# Test
print("\n" + "=" * 70)
print("🔢 POLY1305 IMPLEMENTASİYASI")
print("=" * 70)

# Təsadüfi açar yarat
key = os.urandom(32)
poly = Poly1305(key)

message = b"Salam, dunya! Bu Poly1305 test mesajidir."
tag = poly.hexdigest(message)

print(f"🔑 Açar: {key.hex()}")
print(f"📨 Mesaj: {message}")
print(f"🔏 Poly1305 teq: {tag}")
print("ℹ️ Qeyd: Poly1305 açarı real sistemlərdə hər mesaj üçün yenilənməlidir.")

### 6.1 $r$ açarının məhdudlaşdırılması

<div style="background-color: #f0f8ff; padding: 15px; border-radius: 10px; border-left: 5px solid #4682b4;">
<h4>📌 $r$ açarının məhdudlaşdırılması</h4>
<p>Poly1305-də $r$ açarı xüsusi bir maska ilə məhdudlaşdırılır:</p>
<p style="text-align: center; font-size: 1.1em;">$$r = r \ \&\ \text{0x0ffffffc0ffffffc0ffffffc0fffffff}$$</p>
<p>Bu maska bayt səviyyəsində $r[3]$, $r[7]$, $r[11]$ və $r[15]$ baytlarının yuxarı 4 bitini, $r[4]$, $r[8]$ və $r[12]$ baytlarının isə aşağı 2 bitini sıfırlayır və təhlükəsizliyi artırır.</p>
</div>

In [ ]:
def poly1305_test():
    """
    Poly1305 testi - RFC 7539 / RFC 8439 test vektorları ilə
    """
    print("\n" + "-" * 70)
    print("✅ POLY1305 TEST VEKTORLARI")
    print("-" * 70)

    # RFC 7539 / RFC 8439 test vektoru
    key = bytes.fromhex(
        "85d6be7857556d337f4452fe42d506a8"
        "0103808afb0db2fd4abff6af4149f51b"
    )
    message = b"Cryptographic Forum Research Group"
    expected = "a8061dc1305136c6c22b8baf0c0127a9"

    poly = Poly1305(key)
    tag = poly.hexdigest(message)

    print(f"📌 Test açarı: {key.hex()}")
    print(f"📨 Test mesajı: {message}")
    print(f"\n📊 Gözlənilən teq: {expected}")
    print(f"📊 Alınan teq:    {tag}")

    if CRYPTOGRAPHY_AVAILABLE:
        from cryptography.hazmat.primitives.poly1305 import Poly1305 as CryptoPoly1305
        library_tag = CryptoPoly1305.generate_tag(key, message).hex()
        print(f"📚 Cryptography teqi: {library_tag}")
        print(f"✅ Kitabxana ilə eynidir? {tag == library_tag}")

    print(f"\n✅ RFC test uğurlu? {tag == expected}")

poly1305_test()

### ✍️ Çalışma 2: Poly1305 (1.5 bal)

Aşağıdakı tapşırıqları yerinə yetirin:

1. **Poly1305 testi:** Poly1305 sinfinizi istifadə edərək bir mesajın teqini hesablayın.

2. **$r$ məhdudlaşdırılması:** $r$ açarının məhdudlaşdırılmasının səbəbini izah edin.

3. **Müxtəlif uzunluqlar:** Müxtəlif uzunluqlu mesajlar üçün Poly1305 teqlərini müqayisə edin.

4. **RFC testi:** RFC 7539-dakı test vektorlarını yoxlayın.

In [ ]:
# Çalışma 2 - Cavablar

print("📝 ÇALIŞMA 2 CAVABLARI")
print("=" * 80)

# 1. Poly1305 testi
print("\n1. POLY1305 TESTİ:")
key = os.urandom(32)
poly = Poly1305(key)
message = b"Test mesaji"
tag = poly.hexdigest(message)
print(f"   Açar: {key.hex()}")
print(f"   Mesaj: {message}")
print(f"   Poly1305 teq: {tag}")

# 2. r məhdudlaşdırılması
print("\n2. r MƏHDUDLAŞDIRILMASI:")
print("""
   r açarının məhdudlaşdırılması təhlükəsizlik üçün vacibdir:
   • Müəyyən bitlər sıfırlanır ki, hesablama RFC-yə uyğun olsun
   • Bu, Poly1305-in riyazi təhlükəsizlik sərhədlərini qoruyur
   • Həddən artıq böyük aralıq qiymətlərin qarşısını alır
""")

# 3. Müxtəlif uzunluqlar
print("\n3. MÜXTƏLİF UZUNLUQLAR:")
messages = [b"A", b"AB" * 8, b"ABC" * 16]
for msg in messages:
    one_time_key = os.urandom(32)
    tag = Poly1305(one_time_key).hexdigest(msg)
    print(
        f"   Mesaj ({len(msg)} bayt): {msg[:16]}... | "
        f"Açar (ilk 8 bayt): {one_time_key.hex()[:16]}... -> {tag[:16]}..."
    )
print("   ℹ️ Qeyd: Poly1305 açarı hər mesaj üçün yenilənməlidir.")

# 4. RFC testi (artıq yuxarıda edildi)
print("\n4. RFC TESTİ:")
print("   Yuxarıdakı poly1305_test() funksiyası RFC testlərini yoxladı.")

## 🔐 ChaCha20-Poly1305 AEAD rejimi (20 dəq)

<div style="background-color: #f0f8ff; padding: 15px; border-radius: 10px; border-left: 5px solid #4682b4;">
<h4>🔐 ChaCha20-Poly1305 AEAD rejimi</h4>
<p>ChaCha20-Poly1305 AEAD rejimi iki alqoritmi birləşdirir:</p>
<ul>
    <li><b>ChaCha20</b> - məxfilik təmin edən axın şifrəsi</li>
    <li><b>Poly1305</b> - tamlıq təmin edən MAK</li>
</ul>
</div>

In [ ]:
def chacha20_poly1305_demo():
    """
    ChaCha20-Poly1305 AEAD rejiminin nümayişi
    """
    if not CRYPTOGRAPHY_AVAILABLE:
        print("cryptography kitabxanası yüklü deyil")
        return

    from cryptography.hazmat.primitives.ciphers.aead import ChaCha20Poly1305
    from cryptography.exceptions import InvalidTag

    print("\n" + "-" * 70)
    print("🔐 CHACHA20-POLY1305 AEAD NÜMAYİŞİ")
    print("-" * 70)

    # Açar (32 bayt) və nonce (12 bayt)
    key = ChaCha20Poly1305.generate_key()
    nonce = os.urandom(12)

    print(f"🔑 Açar: {key.hex()}")
    print(f"🔢 Nonce: {nonce.hex()}")

    # Məlumat
    plaintext = "Bu çox gizli mesajdır. Heç kim oxuya bilməz.".encode("utf-8")
    aad = "Əlaqəli məlumat (AAD) - şifrələnmir, amma tamlığı qorunur".encode("utf-8")

    print(f"\n📨 Açıq mətn: {plaintext.decode('utf-8')}")
    print(f"📎 AAD: {aad.decode('utf-8')}")

    # Şifrləmə
    chacha = ChaCha20Poly1305(key)
    ciphertext_with_tag = chacha.encrypt(nonce, plaintext, aad)
    ciphertext, tag = ciphertext_with_tag[:-16], ciphertext_with_tag[-16:]

    print(f"\n🔒 Şifrəli mətn: {ciphertext.hex()}")
    print(f"🏷️ Teq: {tag.hex()}")
    print(f"📏 Şifrəli hissə: {len(ciphertext)} bayt + 16 bayt teq")

    # Deşifrləmə
    try:
        decrypted = chacha.decrypt(nonce, ciphertext_with_tag, aad)
        print(f"\n🔓 Deşifrələnmiş: {decrypted.decode('utf-8')}")
        print("✅ Autentifikasiya UĞURLU!")
    except InvalidTag:
        print("❌ Autentifikasiya XƏTASI! Açar, nonce, AAD və ya şifrəli mətn dəyişdirilib.")

if CRYPTOGRAPHY_AVAILABLE:
    chacha20_poly1305_demo()

### 7.1 Açar və nonce idarəetməsi

In [ ]:
def nonce_management_demo():
    """
    ChaCha20-Poly1305-də nonce idarəetməsi
    """
    print("\n" + "-" * 70)
    print("🔢 NONCE İDARƏETMƏSİ")
    print("-" * 70)

    print("""
    ChaCha20-Poly1305-də nonce (12 bayt):
    • Hər mesaj üçün unikal olmalıdır
    • Eyni açarla nonce təkrarlanarsa, təhlükəsizlik çökür
    • Əsas tövsiyə: sayğac və ya sabit prefiks + sayğac ilə unikal nonce
    • Təsadüfi 96-bit nonce praktikada kolliziya riski yaradır; çox mesaj üçün
      birthday bound nəzərə alınmalıdır və RFC 8439 bu konstruksiya üçün
      nonce-ların unikal seçilməsini tələb edir

    Tövsiyə olunan üsullar:
    1. Sayğac əsaslı: 0, 1, 2, ... (vəziyyət saxlanmalı)
    2. Qarışıq: sabit prefiks + sayğac
    3. Təsadüfi nonce yalnız məhdud riskli mühəndislik seçimi kimi düşünülməlidir;
       təsadüfi nonce dizaynı lazımdırsa, XChaCha20-Poly1305 kimi daha uzun nonce-li
       variantlara baxın
    """)

nonce_management_demo()


In [ ]:
def chacha20_pycryptodome_demo():
    """
    PyCryptodome ilə ChaCha20 axın şifrəsi.

    Xəbərdarlıq: bu demo yalnız məxfiliyi göstərir. Raw ChaCha20
    autentifikasiya təmin etmir və MAC/AEAD olmadan dəyişdirilə biləndir
    (malleable). Real tətbiqlərdə Encrypt-then-MAC və ya ChaCha20-Poly1305
    kimi AEAD konstruksiyasından istifadə edin.
    """
    if not CRYPTO_AVAILABLE:
        return

    print("\n" + "-" * 70)
    print("📦 PYCRYPTODOME İLƏ CHACHA20")
    print("-" * 70)
    print("⚠️  Xəbərdarlıq: raw ChaCha20 autentifikasiya vermir; MAC və ya AEAD lazımdır.")

    from Crypto.Cipher import ChaCha20

    key = get_random_bytes(32)
    nonce = get_random_bytes(12)  # RFC 8439 uyğunluğu üçün 12 bayt

    plaintext = b"Salam, dunya! Bu ChaCha20 testidir."

    cipher = ChaCha20.new(key=key, nonce=nonce)
    ciphertext = cipher.encrypt(plaintext)

    print(f"🔑 Açar: {key.hex()}")
    print(f"🔢 Nonce: {nonce.hex()}")
    print("ℹ️ Qeyd: PyCryptodome ChaCha20 üçün 8 və 12 bayt nonce qəbul edir; burada 12 bayt seçilib.")
    print(f"\n📨 Açıq mətn: {plaintext}")
    print(f"🔒 Şifrəli: {ciphertext.hex()}")

    cipher2 = ChaCha20.new(key=key, nonce=nonce)
    decrypted = cipher2.decrypt(ciphertext)

    print(f"🔓 Deşifrə: {decrypted}")
    print(f"✅ Uğurlu? {plaintext == decrypted}")

if CRYPTO_AVAILABLE:
    chacha20_pycryptodome_demo()


### ✍️ Çalışma 3: ChaCha20-Poly1305 (1.5 bal)

Aşağıdakı tapşırıqları yerinə yetirin:

1. **Fayl şifrləmə:** ChaCha20-Poly1305 istifadə edərək bir faylı şifrləyin və deşifrləyin.

2. **Nonce təkrarı testi:** Eyni nonce iki dəfə istifadə edildikdə nə baş verdiyini test edin.

3. **AAD testi:** AAD dəyişdirildikdə autentifikasiyanın uğursuz olduğunu yoxlayın.

4. **Nəzəri sual:** Nonce təkrarının təhlükəsizliyə təsirini izah edin.

In [ ]:
# Çalışma 3 - Cavablar

print("📝 ÇALIŞMA 3 CAVABLARI")
print("=" * 80)

if CRYPTOGRAPHY_AVAILABLE:
    from cryptography.hazmat.primitives.ciphers.aead import ChaCha20Poly1305
    from cryptography.exceptions import InvalidTag

    # 1. Fayl şifrləmə və deşifrləmə
    print("\n1. FAYL ŞİFRLƏMƏ VƏ DEŞİFRLƏMƏ:")

    def encrypt_file(filename, key, aad=b''):
        nonce = os.urandom(12)
        chacha = ChaCha20Poly1305(key)

        with open(filename, 'rb') as f:
            data = f.read()

        ciphertext = chacha.encrypt(nonce, data, aad)
        out_filename = filename + '.enc'

        with open(out_filename, 'wb') as f:
            f.write(nonce + ciphertext)

        return out_filename

    def decrypt_file(filename, key, aad=b''):
        with open(filename, 'rb') as f:
            raw = f.read()

        nonce, ciphertext = raw[:12], raw[12:]
        chacha = ChaCha20Poly1305(key)
        plaintext = chacha.decrypt(nonce, ciphertext, aad)

        out_filename = filename.removesuffix('.enc') + '.dec'
        with open(out_filename, 'wb') as f:
            f.write(plaintext)

        return out_filename

    file_key = ChaCha20Poly1305.generate_key()
    file_aad = b'lecture17-file-demo'
    sample_filename = 'sample_message.txt'
    original_data = "ChaCha20-Poly1305 fayl testi".encode('utf-8')

    with open(sample_filename, 'wb') as f:
        f.write(original_data)

    enc_filename = encrypt_file(sample_filename, file_key, file_aad)
    dec_filename = decrypt_file(enc_filename, file_key, file_aad)

    with open(dec_filename, 'rb') as f:
        restored_data = f.read()

    print(f"   Orijinal fayl: {sample_filename}")
    print(f"   Şifrəli fayl: {enc_filename}")
    print(f"   Bərpa olunmuş fayl: {dec_filename}")
    print(f"   ✅ Məlumat bərpa edildi? {restored_data == original_data}")

    # 2. Nonce təkrarı testi
    print("\n2. NONCE TƏKRARI TESTİ:")
    key = ChaCha20Poly1305.generate_key()
    nonce = os.urandom(12)

    m1 = b"Message A"
    m2 = b"Message B"

    chacha = ChaCha20Poly1305(key)
    c1 = chacha.encrypt(nonce, m1, b'')
    c2 = chacha.encrypt(nonce, m2, b'')

    xor_ct = bytes(a ^ b for a, b in zip(c1[:-16], c2[:-16]))
    xor_pt = bytes(a ^ b for a, b in zip(m1, m2))

    print(f"   c1 (tag daxil): {c1.hex()}")
    print(f"   c2 (tag daxil): {c2.hex()}")
    print(f"   Şifrəli hissələrin XOR-u = açıq mətnlərin XOR-u? {xor_ct == xor_pt}")
    print("   ⚠ Eyni açar+nonce cütü axın şifrəsində keystream təkrarına səbəb olur.")
    print("   ⚠ Üstəlik, Poly1305 one-time key də təkrar olunur və bu, saxtalaşdırma riskini artırır.")

    # 3. AAD testi
    print("\n3. AAD TESTİ:")
    key = ChaCha20Poly1305.generate_key()
    nonce = os.urandom(12)
    aad = b'header'
    ciphertext = ChaCha20Poly1305(key).encrypt(nonce, b"gizli mesaj", aad)

    try:
        ChaCha20Poly1305(key).decrypt(nonce, ciphertext, b'wrong-header')
        print("   ❌ Xəta: yanlış AAD qəbul edildi")
    except InvalidTag:
        print("   ✅ Yanlış AAD ilə autentifikasiya uğursuz oldu (InvalidTag)")
else:
    print("cryptography kitabxanası yüklü deyil, buna görə praktik testlər keçilmədi.")

# 4. Nəzəri sual
print("\n4. NONCE TƏKRARININ TƏSİRİ:")
print("""
   Nonce təkrarı ChaCha20-Poly1305-də təhlükəlidir, çünki:
   • Eyni açar və nonce eyni açar axını yaradır
   • Buna görə şifrəli hissələrin XOR-u açıq mətnlərin XOR-unu sızdıra bilər
   • Həmçinin Poly1305 üçün eyni one-time key təkrarlandığından saxtalaşdırma riski artır

   Qorunma üsulları:
   • Hər mesaj üçün unikal nonce istifadə edin
   • Sayğac əsaslı nonce saxlayın
   • Təsadüfi nonce-lər üçün toqquşma riskini miqyasdan asılı şəkildə qiymətləndirin
""")

## 📊 HMAC və Poly1305 müqayisəsi (10 dəq)

In [ ]:
def hmac_vs_poly1305_comparison():
    """
    HMAC və Poly1305 müqayisəsi
    """
    print("\n" + "-" * 80)
    print("📊 HMAC vs POLY1305 MÜQAYİSƏSİ")
    print("-" * 80)

    print("""
┌──────────────────┬────────────────────────────┬────────────────────────────┐
│ Xüsusiyyət       │ HMAC                        │ Poly1305                   │
├──────────────────┼────────────────────────────┼────────────────────────────┤
│ Əsas mexanizm    │ Heş funksiyaları            │ Universal heşləmə          │
│ Riyazi əsas      │ Merkl-Damqord / Süngər      │ 𝔽_{2^{130}-5}             │
│ Açar tələbi      │ Bir açar, çox mesaj         │ Hər mesaj üçün yeni açar   │
│ Açar uzunluğu    │ Heşdən asılı (256-512 bit)  │ 256-bit (r+s)              │
│ Çıxış uzunluğu   │ Heşdən asılı (160-512 bit)  │ 128-bit                    │
│ Proqram sürəti   │ Orta - Yüksək               │ Çox yüksək                 │
│ Paralelləşdirmə  │ Məhdud                      │ Asan                       │
│ AEAD dəstəyi     │ Ayrıca kombinasiya          │ ChaCha20 ilə inteqrasiya   │
└──────────────────┴────────────────────────────┴────────────────────────────┘
""")

hmac_vs_poly1305_comparison()

<div style="background-color: #e8f5e9; padding: 15px; border-radius: 10px; border-left: 5px solid #4caf50;">
<h4>📌 Nə vaxt HMAC, nə vaxt Poly1305 seçməli?</h4>
<ul>
    <li><b>HMAC seçin:</b>
        <ul>
            <li>Mövcud sistemlərlə uyğunluq tələb olunarsa</li>
            <li>Heş funksiyaları (SHA-2, SHA-3) artıq istifadə olunursa</li>
            <li>Uzunmüddətli açar lazımdırsa</li>
        </ul>
    </li>
    <li><b>Poly1305 seçin:</b>
        <ul>
            <li>Yüksək sürət tələb olunarsa</li>
            <li>ChaCha20 artıq istifadə olunursa</li>
            <li>Mobil cihazlar / AES-NI olmayan platformalar</li>
            <li>Yeni sistemlər qurulursa</li>
        </ul>
    </li>
</ul>
</div>

## 🖥️ İnteqrasiya edilmiş tətbiq (15 dəq)

In [ ]:
def mac_lab_menu():
    """
    MAK laboratoriyası interaktiv menyu - Məşğələ 17
    """
    available_algorithms = {
        'sha256': hashlib.sha256,
        'sha512': hashlib.sha512,
        'sha3_256': hashlib.sha3_256,
    }

    while True:
        print("\n" + "=" * 70)
        print("🔏 HMAC & POLY1305 LABORATORİYASI - Məşğələ 17")
        print("=" * 70)
        print("1. 🔏 HMAC hesabla")
        print("2. ⚠️ HMAC vs H(K||M) müqayisəsi")
        print("3. 🔢 Poly1305 hesabla (öz implementasiya)")
        print("4. 🔐 ChaCha20-Poly1305 AEAD (cryptography)")
        print("5. 📦 ChaCha20 axın şifrəsi (PyCryptodome)")
        print("6. 🔑 HKDF açar törətmə")
        print("7. ⚡ Performans müqayisəsi")
        print("8. 📊 HMAC vs Poly1305 müqayisəsi")
        print("0. ❌ Çıxış")
        print("=" * 70)

        choice = input("📌 Seçiminiz: ")

        if choice == '1':
            message = input("📨 Mesaj: ").encode()
            key = input("🔑 Açar: ").encode() or b'gizli'
            algo = (input("Heş alqoritmi (sha256/sha512/sha3_256): ") or 'sha256').strip()

            hash_func = available_algorithms.get(algo)
            if hash_func is None:
                print("❌ Dəstəklənən alqoritmlər: sha256, sha512, sha3_256")
                continue

            h = hmac.new(key, message, hash_func)
            print(f"\n🔏 HMAC-{algo}: {h.hexdigest()}")

        elif choice == '2':
            length_extension_attack_demo()

        elif choice == '3':
            message = input("📨 Mesaj: ").encode()
            key = os.urandom(32)
            poly = Poly1305(key)
            tag = poly.hexdigest(message)

            print(f"\n🔑 Açar: {key.hex()}")
            print(f"🔏 Poly1305 teq: {tag}")
            print("ℹ️ Qeyd: bu açar yalnız bir mesaj üçün istifadə olunmalıdır.")

        elif choice == '4' and CRYPTOGRAPHY_AVAILABLE:
            chacha20_poly1305_demo()

        elif choice == '5' and CRYPTO_AVAILABLE:
            chacha20_pycryptodome_demo()

        elif choice == '6':
            hkdf_demo()

        elif choice == '7':
            import time

            data = os.urandom(1024 * 1024)  # 1 MB
            key = b'gizli'

            # HMAC-SHA256
            start = time.time()
            h = hmac.new(key, data, hashlib.sha256)
            h.digest()
            hmac_time = time.time() - start

            # Poly1305 (öz implementasiya)
            poly_key = os.urandom(32)
            poly = Poly1305(poly_key)
            start = time.time()
            poly.hash(data)
            poly_time = time.time() - start

            print(f"\n📊 1 MB məlumat üçün:")
            print(f"   HMAC-SHA256: {hmac_time*1000:.2f} ms")
            print(f"   Poly1305:    {poly_time*1000:.2f} ms")
            print(f"   Sürət nisbəti: {hmac_time/poly_time:.2f}x")

        elif choice == '8':
            hmac_vs_poly1305_comparison()

        elif choice == '0':
            print("👋 Proqram bitdi. Sağ olun!")
            break

        else:
            print("❌ Yanlış seçim və ya kitabxana yüklü deyil")

# Proqramı işə sal (istəyə bağlı)
# mac_lab_menu()

## 🏠 Ev tapşırığı

### 📦 Ev tapşırığı 1: MAK paketi (3 bal)

Aşağıdakı funksiyaları özündə birləşdirən Python paketi yazın:

```
mac_package/
├── __init__.py
├── hmac_demo.py        # HMAC hesablamaları, uzunluq genişləndirmə hücumu
├── poly1305.py         # Poly1305 implementasiyası (öz sinfiniz)
├── chacha20_poly1305.py # ChaCha20-Poly1305 AEAD nümayişi
├── hkdf.py             # HKDF açar törətmə funksiyası
├── comparison.py       # HMAC vs Poly1305 müqayisəsi
└── main.py             # Bütün funksiyaları birləşdirən interaktiv menyu
```

**Tələblər:**
* Hər bir funksiya üçün docstring yazın
* Səhv hallarını idarə edin (try-except)
* Kod təmiz və oxunaqlı olmalıdır
* Hər modul üçün test nümunələri əlavə edin

### 🔐 Ev tapşırığı 2: Praktik məsələlər (2 bal)

Aşağıdakı məsələləri həll edin:

1. **API autentifikasiya:** HMAC-SHA256 istifadə edərək bir API autentifikasiya sistemi qurun (məsələn, AWS Signature-a bənzər).

2. **Poly1305 testi:** Poly1305 implementasiyanızı RFC 8439-dakı test vektorları ilə yoxlayın.

3. **Böyük fayl:** ChaCha20-Poly1305 ilə böyük bir faylı (10 MB) şifrləyin və deşifrləyin.

4. **Performans qrafiki:** HMAC və Poly1305-in performansını müqayisə edən qrafik çəkin.

### 📚 Ev tapşırığı 3: Tədqiqat (2 bal)

Araşdırma aparın və aşağıdakı suallara cavab tapın. Cavablarınızı 1-2 səhifəlik hesabat şəklində təqdim edin:

1. **HMAC yaradıcıları:** Mihir Bellare, Ran Canetti, Hugo Krawczyk kimdir? HMAC-ın yaradılmasında rolları nədir?

2. **Daniel J. Bernstein:** Daniel J. Bernstein kimdir? Onun kriptoqrafiyaya töhfələri nələrdir? (ChaCha20, Poly1305, Curve25519)

3. **TLS 1.3 rejimləri:** TLS 1.3-də niyə həm AES-GCM, həm də ChaCha20-Poly1305 dəstəklənir? Hansı vəziyyətlərdə hansı üstünlük təşkil edir?

4. **WireGuard seçimi:** WireGuard VPN protokolu niyə məhz ChaCha20-Poly1305 seçib? AES-in problemləri nədir?

**Format tələbləri:**
* PDF formatında təqdim edin
* Ən azı 5 qaynaq göstərin (kitab, məqalə, veb səhifə)
* Öz sözlərinizlə yazın (copy-paste yox)
* Mümkünsə, sxemlər və ya diaqramlar əlavə edin

## 📌 Yekun və müzakirə sualları

<div style="background-color: #e8f4f8; padding: 15px; border-radius: 10px; border-left: 5px solid #2c3e50;">
<h3>📋 Xülasə</h3>
<p>Bu məşğələdə öyrəndiklərimiz:</p>
<ul>
    <li>✅ <b>HMAC:</b> Heş funksiyaları əsasında qurulan MAK. $H((K \oplus \text{opad}) \parallel H((K \oplus \text{ipad}) \parallel M))$</li>
    <li>✅ <b>Uzunluq genişləndirmə hücumu:</b> $H(K || M)$ konstruksiyasının zəifliyi. HMAC bu hücuma davamlıdır.</li>
    <li>✅ <b>Universal heşləmə:</b> Təsadüfi açar üçün toqquşma ehtimalının çox aşağı olduğu heş funksiyaları ailəsi.</li>
    <li>✅ <b>Poly1305:</b> $\mathbb{F}_{2^{130}-5}$ meydanında işləyən universal MAK. Hər mesaj üçün yeni açar tələb edir.</li>
    <li>✅ <b>ChaCha20-Poly1305:</b> ChaCha20 şifrələmə + Poly1305 autentifikasiya. TLS 1.3, WireGuard, SSH-də istifadə olunur.</li>
    <li>✅ <b>HKDF:</b> HMAC əsaslı açar törətmə funksiyası (RFC 5869).</li>
</ul>
<p><i>Praktik tövsiyə: Mövcud sistemlər üçün HMAC-SHA256, yeni sistemlər və mobil cihazlar üçün ChaCha20-Poly1305 seçin.</i></p>
</div>

### 💭 Müzakirə sualları

1. Bu məşğələdə ən maraqlı tapdığınız nə oldu?
2. HMAC-ın $H(K || M)$-dən üstünlüyü nədir?
3. Poly1305 niyə hər mesaj üçün yeni açar tələb edir?
4. ChaCha20-Poly1305 niyə TLS 1.3-də AES-GCM-ə alternativ olaraq seçilib?
5. Universal heşləmə ilə klassik heş funksiyaları arasında nə fərq var?
6. HKDF-nin extract və expand mərhələləri nə üçün vacibdir?
7. Nonce təkrarı niyə ChaCha20-Poly1305-də təhlükəlidir?

## 📚 Əlavə resurslar

* 📘 **RFC 2104 (HMAC):** [https://tools.ietf.org/html/rfc2104](https://tools.ietf.org/html/rfc2104)
* 📙 **RFC 7539 (ChaCha20-Poly1305):** [https://tools.ietf.org/html/rfc7539](https://tools.ietf.org/html/rfc7539)
* 📗 **RFC 8439 (yenilənmiş ChaCha20-Poly1305):** [https://tools.ietf.org/html/rfc8439](https://tools.ietf.org/html/rfc8439)
* 📕 **RFC 5869 (HKDF):** [https://tools.ietf.org/html/rfc5869](https://tools.ietf.org/html/rfc5869)
* 📘 **Python hmac modulu:** [https://docs.python.org/3/library/hmac.html](https://docs.python.org/3/library/hmac.html)
* 📙 **Cryptography kitabxanası:** [https://cryptography.io/](https://cryptography.io/)
* 📗 **Daniel J. Bernstein-in səhifəsi:** [https://cr.yp.to/djb.html](https://cr.yp.to/djb.html)
* 📘 **WireGuard White Paper:** [https://www.wireguard.com/papers/wireguard.pdf](https://www.wireguard.com/papers/wireguard.pdf)

---

<div style="background-color: #f0f0f0; padding: 20px; border-radius: 10px; text-align: center;">
<h2>✅ Məşğələ 17 tamamlandı!</h2>
<p>Bütün kodları və tapşırıq cavablarını növbəti məşğələyə qədər təqdim edin.</p>
<p><em>Kodlar aydın şərhlərlə yazılmalı və asan oxunmalıdır. Hər bir funksiyanın nə etdiyini izah edən şərhlər əlavə edin.</em></p>
<p style="font-size: 1.2em; margin-top: 15px;">🔏 <b>HMAC və Poly1305 - məlumat tamlığının qoruyucuları!</b></p>
</div>